# 2. Variabeltyper och skalnivåer

Det första man gör med ny data är att fråga sig: **vad är det för sorts värden?**
Svaret avgör vilka beräkningar, diagram och tester som är meningsfulla. En dator räknar gärna ut medelvärdet
av postnummer – det är du som måste veta att svaret är nonsens.

**Efter den här notebooken kan du**
- skilja på kvalitativa och kvantitativa variabler, och på diskreta och kontinuerliga
- avgöra en variabels skalnivå: nominal, ordinal, intervall eller kvot
- välja ett lägesmått som passar skalnivån
- se skillnaden mellan hur pandas *lagrar* en kolumn och vad den *betyder*

**Förkunskaper:** [1. Verktygen](../01_verktygen/01_verktygen.ipynb)

**OBS!** Variabeltyper inom programmering och statistik betyder inte samma sak! I programmering handlar det om hur datorn lagrar värden, medan statistik fokuserar på vad värdena representerar och hur de kan analyseras. *Det vi pratar om här är alltså statistiska variabeltyper.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 2.1 En liten datamängd

Åtta kunder i en webbutik. Titta på kolumnerna: vilka innehåller tal man kan räkna med, och vilka gör det inte?

In [ ]:
customers = pd.DataFrame({
    "postcode":     [22100, 90325, 11420, 22100, 62157, 41103, 90325, 11420],
    "channel":      ["App", "Webb", "App", "Butik", "Webb", "App", "App", "Butik"],
    "membership":   ["Silver", "Guld", "Brons", "Silver", "Guld", "Guld", "Silver", "Brons"],     # medlemsnivå
    "satisfaction": [4, 5, 2, 3, 5, 4, 1, 3],          # betyg 1–5
    "num_orders":   [3, 12, 1, 5, 8, 2, 0, 4],
    "amount":       [349.0, 1250.5, 99.0, 720.0, 910.9, 215.5, 0.0, 480.0],   # kr
})
customers

## 2.2 Kvalitativ eller kvantitativ?

| Typ | Beskrivning | Exempel ovan |
|---|---|---|
| **Kvalitativ** (kategorisk) | Värdena är kategorier, inte mängder | `channel`, `membership` |
| **Kvantitativ, diskret** | Räknas i hela steg | `num_orders` |
| **Kvantitativ, kontinuerlig** | Kan anta vilket värde som helst i ett intervall | `amount` |

Fällan är att **siffror inte alltid är tal**. Vad händer om vi ber pandas om medelvärdet av allt som ser ut som tal?

🔮 **Gissa först:** vilka av medelvärdena nedan är meningsfulla?

In [ ]:
customers[["postcode", "satisfaction", "num_orders", "amount"]].mean()

- Medelvärdet av `postcode` går att räkna ut men **betyder ingenting**. Postnummer är etiketter som råkar bestå av siffror.
  Variabeln är kvalitativ.
- `num_orders` och `amount` är äkta mängder, och där fungerar medelvärdet.
- `satisfaction` är ett gränsfall som vi återkommer till strax.

Testet är: *är det meningsfullt att addera två värden?* 3 order + 5 order = 8 order.
Men postnummer 22100 + 90325 är ingenting.

## 2.3 Skalnivåer

Skalnivån är en finare indelning som talar om **hur mycket information** värdena bär.

| Skalnivå | Vad kan man säga? | Exempel | Lämpligt lägesmått |
|---|---|---|---|
| **Nominalskala** | Bara lika eller olika | `channel`, `postcode` | Typvärde |
| **Ordinalskala** | Dessutom ordning (mer/mindre), men avstånden är inte bestämda | `membership`, `satisfaction` | Typvärde, median |
| **Intervallskala** | Dessutom lika stora steg, men ingen äkta nollpunkt | temperatur i °C, årtal | Medelvärde, median |
| **Kvotskala** | Dessutom äkta nollpunkt, så "dubbelt så mycket" är meningsfullt | `num_orders`, `amount` | Medelvärde, median |

Varje nivå klarar allt som nivåerna ovanför klarar, plus något mer.

**Intervall eller kvot?** Fråga: *betyder noll "ingenting"?* 0 kr är inga pengar, så belopp är kvotskala och
200 kr är dubbelt så mycket som 100 kr. Men 0 °C betyder inte "ingen temperatur", och 20 °C är inte dubbelt så varmt som 10 °C.

**Varför är `satisfaction` ordinal?** Betyg 4 är bättre än 2, men vi vet inte om steget från 1 till 2 är lika stort som
från 4 till 5. I praktiken räknar man ändå ofta medelvärde på sådana skalor – var då medveten om antagandet.

### Lägesmått som passar skalnivån

Tre mått som beskriver ett "typiskt" värde (mer om dem i nästa notebook):

In [ ]:
print("Typvärde för channel:     ", customers["channel"].mode()[0])        # vanligaste värdet – fungerar alltid
print("Median för satisfaction:  ", customers["satisfaction"].median())   # kräver ordning
print("Medelvärde för amount:    ", customers["amount"].mean())           # kräver lika stora steg

🧪 **Labba:** Pröva `customers["channel"].median()` i cellen nedan. Vad händer, och varför är felmeddelandet rimligt?

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Svar</b></summary>

Du får ett fel (`TypeError`). En median kräver att värdena kan sorteras från minst till störst, och
"App", "Webb" och "Butik" har ingen sådan ordning. Här säger pandas ifrån – men för `postcode` gör den inte det.

</details>

## 2.4 pandas vet inte vad dina variabler betyder

`dtypes` visar hur pandas **lagrar** varje kolumn. Det är inte samma sak som variabeltypen.

In [ ]:
customers.dtypes

`postcode` lagras som heltal, precis som `num_orders`, trots att den ena är nominal och den andra kvot.
Och `membership` lagras som text, så pandas känner inte till att Brons < Silver < Guld. Se vad som händer när vi sorterar:

In [ ]:
customers["membership"].value_counts().sort_index()

Ordningen blev alfabetisk, med Guld mellan Brons och Silver. Vi kan berätta för pandas att variabeln är **ordinal** genom att ange kategoriernas ordning:

In [ ]:
levels = ["Brons", "Silver", "Guld"]
customers["membership"] = pd.Categorical(customers["membership"], categories=levels, ordered=True)

print(customers["membership"].value_counts().sort_index())
print()
print("Högsta nivå i datan:", customers["membership"].max())
print("Antal över bronsnivå:", (customers["membership"] > "Brons").sum())

Nu fungerar jämförelser som `>` och `max()`, eftersom pandas känner till ordningen.

✏️ **Din tur:** Gör ett stapeldiagram över `membership` med kategorierna i rätt ordning.
Ledtråd: `customers["membership"].value_counts().sort_index().plot(kind="bar", rot=0)`. Lägg till axelrubriker.

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
customers["membership"].value_counts().sort_index().plot(kind="bar", rot=0)
plt.xlabel("Medlemsnivå")
plt.ylabel("Antal kunder")
plt.show()
```

</details>

## 2.5 Testa dig själv: klassificera variabler

✏️ **Din tur:** Byt ut varje `"?"` mot `"nominal"`, `"ordinal"`, `"intervall"` eller `"kvot"` och kör cellen.
Du får besked direkt om vilka som är rätt.

In [ ]:
def check_answers(answers):
    key = {
        "blodgrupp": ("nominal", "Kategorier utan ordning."),
        "klockslag": ("intervall", "Lika stora steg, men 00:00 betyder inte 'ingen tid', och klockan 14 är inte dubbelt så mycket som klockan 7."),
        "placering i ett lopp": ("ordinal", "Ordning finns, men tidsavståndet mellan 1:a och 2:a kan vara ett helt annat än mellan 2:a och 3:a."),
        "födelseår": ("intervall", "Lika stora steg, men år 0 är en godtycklig nollpunkt."),
        "ålder i år": ("kvot", "Äkta nollpunkt: en 40-åring har levt dubbelt så länge som en 20-åring."),
        "telefonnummer": ("nominal", "Siffror, men bara en etikett."),
        "betyg A–F": ("ordinal", "Ordnade kategorier med obestämda avstånd."),
        "restid i minuter": ("kvot", "Äkta nollpunkt, och 40 minuter är dubbelt så långt som 20."),
    }
    num_correct = 0
    for variable, (correct, explanation) in key.items():
        answer = answers.get(variable, "?")
        if answer == "?":
            print(f"⬜ {variable}: inte ifylld än")
        elif answer == correct:
            num_correct += 1
            print(f"✅ {variable}: {correct}. {explanation}")
        else:
            print(f"❌ {variable}: inte {answer}. Tänk på: finns ordning? lika stora steg? äkta nollpunkt?")
    print(f"\n{num_correct} av {len(key)} rätt")

In [ ]:
answers = {
    "blodgrupp": "?",
    "klockslag": "?",
    "placering i ett lopp": "?",
    "födelseår": "?",
    "ålder i år": "?",
    "telefonnummer": "?",
    "betyg A–F": "?",
    "restid i minuter": "?",
}
check_answers(answers)

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
check_answers({
    "blodgrupp": "nominal",
    "klockslag": "intervall",
    "placering i ett lopp": "ordinal",
    "födelseår": "intervall",
    "ålder i år": "kvot",
    "telefonnummer": "nominal",
    "betyg A–F": "ordinal",
    "restid i minuter": "kvot",
})
```

</details>

## Sammanfattning

- **Kvalitativ** variabel = kategorier. **Kvantitativ** = mängder, antingen **diskreta** (räknas) eller **kontinuerliga** (mäts).
- Skalnivåerna **nominal → ordinal → intervall → kvot** bär allt mer information.
- Skalnivån avgör vad som är meningsfullt: typvärde fungerar alltid, median kräver ordning, medelvärde kräver lika stora steg.
- Datorn kontrollerar inte detta åt dig. Siffror är inte alltid tal.

### Kontrollfrågor
1. En enkät kodar kön som 1 och 2. Någon rapporterar "medelvärde 1,6". Vad är problemet, och vad borde redovisas?
2. Varför är temperatur i Kelvin kvotskala när temperatur i Celsius är intervallskala?

<details>
<summary>💡 <b>Svar</b></summary>

1. Siffrorna är bara etiketter (nominalskala), så medelvärdet saknar mening. Redovisa andelar, till exempel "60 % svarade alternativ 2".
2. 0 K är en äkta nollpunkt (ingen värmerörelse alls), så 200 K är verkligen dubbelt så mycket som 100 K. 0 °C är bara vattnets fryspunkt.

</details>

**Nästa notebook:** [3. Deskriptiv statistik](../03_deskriptiv_statistik/03_deskriptiv_statistik.ipynb)